# Расчетные сетки в FESTIM

FESTIM предоставляет встроенные средства для создания одномерных сеток без использования внешних библиотек, таких как DOLFINx. Такие сетки имеют простую структуру, легко задаются и хорошо подходят для задач, в которых достаточно одномерного приближения (ТДС, проницаемость через материалы и т.д.).

В этом примере мы рассмотрим создание и использование разных типов одномерных сеток в FESTIM:

- **Равномерные сетки**, в которых элементы равномерно распределены по расчётной области  
- **Неравномерные сетки**, которые позволяют локально сгущать сетку для повышения разрешения вблизи точек интереса.

These tools are particularly useful for rapid prototyping, simplified analysis, and cases where full 2D or 3D simulations are not necessary.

By the end of this tutorial, you’ll be able to create and customise 1D meshes directly in FESTIM and use them in a working simulation.

## Равномерные одномерные сетки

Класс `F.Mesh1D` позволяет задавать одномерную сетку, просто указывая координаты её вершин. Таким образом можно легко создавать равномерные или структурированные сетки на заданном интервале.

**Равномерная сетка** — это сетка с постоянным расстоянием между соседними точками. Такой тип сетки хорошо подходит для задач, в которых физические свойства или ожидаемые градиенты решения распределены по области достаточно равномерно.

FESTIM принимает координаты вершин в виде списка чисел или массива NumPy. Ниже показаны два распространённых способа задания равномерной сетки. Первый – с использованием списка:

In [ ]:
import festim as F

simple_mesh = F.Mesh1D(vertices=[0, 1, 2, 3, 4, 5, 6, 7, 7.5])

print(simple_mesh.vertices)

Пример сетки для области от $0$ до $7{,}0\times 10^{-6}$ м с 50 ячейками, заданной с помощью `NumPy`:

In [ ]:
import numpy as np

mesh = F.Mesh1D(vertices=np.linspace(0, 7e-6, num=50))

print(mesh.vertices)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 3))
plt.plot(mesh.vertices, np.zeros_like(mesh.vertices), marker="o", color="tab:blue", alpha=0.5)
plt.xlabel("Координата (м)")
plt.yticks([])
plt.gca().spines[['top', 'right', 'left']].set_visible(False)
plt.show()

**Примечание.** Список координат вершин не обязан начинаться с нуля. Сетка может охватывать любой интервал, например $[1, 20]$ или $[-10^{-4}, 0]$, в зависимости от моделируемой физической области.

In [ ]:
mesh = F.Mesh1D(vertices=np.linspace(1, 20, num=40))

## Неравномерные одномерные сетки

FESTIM также поддерживает **неравномерные одномерные сетки**. Они полезны, когда требуется локально увеличить разрешение сетки в отдельных областях, например вблизи внешних границ, границ раздела или слоёв материалов, где необходима более высокая точность решения.

Чтобы задать неравномерную сетку, достаточно передать список или массив координат вершин с **неравномерным шагом**. Это позволяет полностью управлять разрешением сетки по всей расчётной области.

In [ ]:
# Задаём сетку с локальным сгущением у левой и правой границ
vertices = np.concatenate(
    [
        np.linspace(0, 1e-7, num=200),
        np.linspace(1e-7, 2e-6, num=1000),
        np.linspace(2e-6, 3e-6, num=200),
    ]
)

mesh = F.Mesh1D(vertices=vertices)

Изменение плотности сетки вдоль расчётной области можно наглядно показать с помощью гистограммы координат вершин. Это позволяет убедиться, что в областях локального сгущения сетка действительно имеет более высокое разрешение.

В приведённом ниже примере наблюдается повышенная концентрация вершин вблизи $x = 0$ и $x = 3 \times 10^{-6}$ м и более крупный шаг в средней части области.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(mesh.vertices, density=True, bins=60, alpha=0.3, edgecolor="tab:blue", histtype="stepfilled")
plt.xlabel("Координата (м)")
plt.ylabel("Плотность вершин (1/м)")
plt.gca().spines[['top', 'right']].set_visible(False)
plt.show()

В некоторых задачах удобно начинать с мелкого шага сетки, а затем постепенно увеличивать размеры ячеек, например при моделировании диффузии в направлении от поверхности.

Для этого можно задать сетку с геометрически возрастающими интервалами.

In [ ]:
vertices = np.geomspace(1e-7, 1e-3, num=500)

mesh = F.Mesh1D(vertices=vertices)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

plt.sca(axs[0])
plt.title("Вершины сетки")
plt.plot(mesh.vertices, np.zeros_like(mesh.vertices), marker="o", color="tab:blue", alpha=0.5)

plt.yticks([])
plt.gca().spines[['top', 'right', 'left']].set_visible(False)

plt.sca(axs[1])
plt.hist(mesh.vertices, density=True, bins=50, edgecolor="tab:blue", histtype="step")
plt.xlabel("Координата (м)")
plt.ylabel("Плотность вершин (1/м)")
plt.yscale("log")
plt.gca().spines[['top', 'right']].set_visible(False)
plt.show()

Ниже приведён пример, в котором размеры ячеек увеличиваются на 10% на каждом шаге. Начальная ширина ячейки равна $10^{-7}$ м, а общая длина расчётной области достигает $10^{-3}$ м.

In [ ]:
growth_rate = 1.1
initial_length = 1e-7
total_size = 1e-3
sizes = [initial_length]

while sum(sizes) < total_size:
    sizes.append(sizes[-1] * growth_rate)

# уменьшаем последний размер ячейки, если превышена заданная длина
if sum(sizes) > total_size:
    sizes[-1] = total_size - sum(sizes[:-1])

vertices = np.cumsum(sizes)
mesh = F.Mesh1D(vertices=vertices)


In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

plt.sca(axs[0])
plt.title("Вершины сетки")
plt.plot(mesh.vertices, np.zeros_like(mesh.vertices), marker="o", color="tab:blue", alpha=0.5)
plt.gca().spines[['top', 'right', 'left']].set_visible(False)

plt.yticks([])

plt.sca(axs[1])
plt.hist(mesh.vertices, density=True, bins=50, edgecolor="tab:blue", histtype="step")
plt.xlabel("Координата (м)")
plt.ylabel("Плотность вершин (1/м)")
plt.yscale("log")
plt.gca().spines[['top', 'right']].set_visible(False)
plt.show()